# 🧠🤖 第1周·Day 1 | 自注意力机制核心原理

> **今日目标**: 理解 Self-Attention 的核心思想——每个词去"看"其他所有词，决定自己该关注谁。

---

## 📝 核心知识点

### Self-Attention = 每个词都在"扫视"整个句子

- 输入一个句子，每个词会生成三个向量：**Q（查询）、K（键）、V（值）**
- **Q**：这个词"想找什么"（我在找什么样的信息）
- **K**：这个词"能提供什么"（我能提供什么信息）
- **V**：这个词"实际内容"（我的具体含义）
- 计算方式：Q 和 K 做点积得到**注意力分数** → softmax归一化 → 用分数对 V 加权求和

### 一个具体例子

> "小明喜欢AI，因为**他**对编程感兴趣"

- "他"的 Q 会去和每个词的 K 做匹配
- "他"和"小明"的注意力分数最高（它们指同一个人）
- 所以"他"的输出会大量包含"小明"的 V 信息
- **这就是为什么模型能理解代词指代！**

## 💡 三个比喻理解 Q/K/V

| 角色 | 比喻 | 作用 |
|------|------|------|
| **Q 查询** | 搜索框输入的关键词 | "我在找什么" |
| **K 键** | 网页的标题和标签 | "这个内容关于什么" |
| **V 值** | 网页的实际内容 | "这个内容的详细信息" |
| **注意力** | 搜索结果的相关度排序 | Q·K 越大越相关 |

- Q·K 点积 → 搜索相关性
- softmax → 归一化为概率
- 概率 × V → 加权融合信息

## 🔑 英文术语

- **Query (Q)** [ˈkwɪəri] 查询向量
- **Key (K)** [kiː] 键向量
- **Value (V)** [ˈvæljuː] 值向量
- **Self-Attention** [sɛlf əˈtɛnʃən] 自注意力机制
- **Softmax** [ˈsɒftmæks] 归一化函数
- **Dot Product** [dɒt ˈprɒdʌkt] 点积

## 🎬 推荐视频

> ⭐ **首选**: 3Blue1Brown - 直观解释注意力机制（全球公认最优秀的可视化讲解）
> https://www.bilibili.com/video/BV1TZ421j7Ke/

> 📺 **备选**: 15分钟认识注意力机制（B站·数学原理详解）
> https://www.bilibili.com/video/BV1pj42137ZY/

> 🎓 **进阶**: 李沐 - Attention Is All You Need 论文精读
> https://www.bilibili.com/video/BV1pu411o7BE/

📖 **延伸阅读**：

> ⭐ **首选**: Jay Alammar - The Illustrated Transformer（全网最经典图解，必看！）
> https://jalammar.github.io/illustrated-transformer/

> 📝 **备选**: 图解Transformer：深入理解Self-Attention（知乎）
> https://zhuanlan.zhihu.com/p/651018724

> 💻 **代码**: The Annotated Transformer（Harvard NLP，带详细注释的实现）
> http://nlp.seas.harvard.edu/2018/04/03/attention.html


## 💻 代码演示：手动计算 Self-Attention

让我们用最简单的数字，一步步看清楚 Q/K/V 到底怎么算。

In [2]:
import numpy as np

np.random.seed(42)

# 模拟句子 "我 爱 AI" —— 3个词，每个词4维向量
X = np.array([
    [1.0, 0.0, 0.0, 0.0],   # "我"
    [0.0, 1.0, 0.0, 0.0],   # "爱"
    [0.0, 0.0, 1.0, 0.5],   # "AI"
])

seq_len, d_model = X.shape
print(f"句子长度: {seq_len}, 向量维度: {d_model}")
print(f"输入矩阵 X:\n{X}")


句子长度: 3, 向量维度: 4
输入矩阵 X:
[[1.  0.  0.  0. ]
 [0.  1.  0.  0. ]
 [0.  0.  1.  0.5]]


In [3]:
# Step 1: 生成 Q, K, V
# 实际中 W_q/W_k/W_v 是训练出来的，这里随机初始化
W_q = np.random.randn(d_model, d_model)
W_k = np.random.randn(d_model, d_model)
W_v = np.random.randn(d_model, d_model)

Q = X @ W_q   # 查询："我想找什么"
K = X @ W_k   # 键：  "我能提供什么"
V = X @ W_v   # 值：  "我的实际内容"

print(f"Q:\n{np.round(Q, 3)}\n")
print(f"K:\n{np.round(K, 3)}\n")
print(f"V:\n{np.round(V, 3)}")


Q:
[[ 0.497 -0.138  0.648  1.523]
 [-0.234 -0.234  1.579  0.767]
 [-0.348 -0.414 -1.326 -0.747]]

K:
[[-1.013  0.314 -0.908 -1.412]
 [ 1.466 -0.226  0.068 -1.425]
 [-0.845 -0.035 -1.452  1.302]]

V:
[[-1.300e-02 -1.058e+00  8.230e-01 -1.221e+00]
 [ 2.090e-01 -1.960e+00 -1.328e+00  1.970e-01]
 [-1.000e-03 -1.890e-01 -3.460e-01  2.270e-01]]


In [5]:
# Step 2: 计算注意力分数 (Q × K^T)
scores = Q @ K.T
print("原始注意力分数 (Q·K^T):")
print(np.round(scores, 3))
print("\n解读: 分数越高，两个词越相关")


原始注意力分数 (Q·K^T):
[[-3.286 -1.367  0.628]
 [-2.354 -1.277 -1.088]
 [ 2.482  0.557  1.261]]

解读: 分数越高，两个词越相关


In [6]:
# Step 3: 缩放 + Softmax
d_k = d_model
scaled = scores / np.sqrt(d_k)

def softmax(x):
    x_shifted = x - x.max(axis=1, keepdims=True)  # 防溢出
    exp_x = np.exp(x_shifted)
    return exp_x / exp_x.sum(axis=1, keepdims=True)

attn_weights = softmax(scaled)
print("缩放后的分数:")
print(np.round(scaled, 3))
print("\n注意力权重 (softmax后，每行和=1):")
print(np.round(attn_weights, 3))


缩放后的分数:
[[-1.643 -0.683  0.314]
 [-1.177 -0.639 -0.544]
 [ 1.241  0.279  0.631]]

注意力权重 (softmax后，每行和=1):
[[0.094 0.244 0.662]
 [0.218 0.373 0.41 ]
 [0.519 0.198 0.282]]


In [5]:
# Step 4: 加权求和得到输出
output = attn_weights @ V
print("最终输出:")
print(np.round(output, 3))

print("\n✅ 每个词的输出 = 所有词的值(V)的加权平均")
print("权重 = 注意力分数（softmax后）")


最终输出:
[[ 0.049 -0.702 -0.477  0.084]
 [ 0.075 -1.038 -0.458 -0.099]
 [ 0.034 -0.991  0.066 -0.531]]

✅ 每个词的输出 = 所有词的值(V)的加权平均
权重 = 注意力分数（softmax后）


## 💡 业务关联思考

Self-Attention 本质就是"全方位扫描后做判断"——

就像你分析糖水店经营时，不能只看某一天的销量。你要同时看**季节趋势、单品销量、原材料价格、竞争情况**多个维度，然后综合判断。

Self-Attention 做的就是这件事：让每个词（每个维度）都能"看到"所有其他词（所有其他维度），然后自动决定该关注哪些。

---

> 💡 **进度: W1 Day 1/7 | 🤖 大模型**